# Task 04: Phone-Only Detection — Filter Out the Noise

Same webcam feed, but now we:
1. ONLY draw boxes around phones (ignore people, chairs, etc.)
2. Only show detections above our confidence threshold
3. Show phone count on screen

**Press 'q' to quit.**

## Cell 1: Setup

In [ ]:
from ultralytics import YOLO
import cv2
import time

# Load YOLO
model = YOLO('yolov8n.pt')

# What is the class ID for "cell phone" in YOLO?
# Let's find it:
for class_id, class_name in model.names.items():
    if 'phone' in class_name.lower() or 'cell' in class_name.lower():
        print(f"Found it! Class ID {class_id} = '{class_name}'")
        PHONE_CLASS_ID = class_id

print(f"\nWe will ONLY look for class ID {PHONE_CLASS_ID} (cell phone)")
print("Everything else will be ignored.")

## Cell 2: Set the confidence threshold

This is the minimum confidence to count as a real phone detection.

**Try changing this number** and see what happens:
- Set to 0.3 (30%) → more detections, more false alarms
- Set to 0.5 (50%) → balanced (recommended to start)
- Set to 0.8 (80%) → very strict, might miss some phones

In [ ]:
# ====================================
# CHANGE THIS NUMBER TO EXPERIMENT!
# ====================================
CONFIDENCE_THRESHOLD = 0.5   # 50% — change to 0.3 or 0.8 and re-run to see the difference

print(f"Confidence threshold set to: {CONFIDENCE_THRESHOLD*100:.0f}%")
print(f"Only showing phone detections above {CONFIDENCE_THRESHOLD*100:.0f}% confidence.")

## Cell 3: Live phone-only detection

**What's different from Task 03:**
- We DON'T use `results[0].plot()` (that draws ALL detections)
- Instead, we manually check each detection:
  - Is it a phone? → YES → draw box
  - Is confidence above threshold? → YES → draw box
  - Otherwise → skip it

**Place your phone on desk, then run this cell. Press 'q' to quit.**

In [ ]:
camera = cv2.VideoCapture(0)

if not camera.isOpened():
    print("ERROR: Cannot open webcam!")
else:
    print("Webcam opened! Detecting PHONES ONLY...")
    print("Press 'q' to quit.")
    
    frame_count = 0
    start_time = time.time()
    
    while True:
        success, frame = camera.read()
        if not success:
            break
        
        # Run YOLO detection
        results = model(frame, verbose=False)
        
        # Count phones detected in this frame
        phone_count = 0
        
        # Check EACH detection one by one
        for box in results[0].boxes:
            class_id = int(box.cls[0])        # What object is it?
            confidence = float(box.conf[0])    # How confident?
            
            # FILTER: Only phones + above threshold
            if class_id == PHONE_CLASS_ID and confidence >= CONFIDENCE_THRESHOLD:
                phone_count += 1
                
                # Get box coordinates (where to draw the rectangle)
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
                # Draw a RED rectangle around the phone
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                
                # Draw label above the box
                label = f"PHONE {confidence*100:.0f}%"
                cv2.putText(frame, label, (x1, y1 - 10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        
        # Show phone count and status on screen
        if phone_count > 0:
            status = f"ALERT: {phone_count} PHONE(S) DETECTED!"
            color = (0, 0, 255)  # Red
        else:
            status = "No phones detected"
            color = (0, 255, 0)  # Green
        
        cv2.putText(frame, status, (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        
        # Show FPS
        frame_count += 1
        elapsed = time.time() - start_time
        fps = frame_count / elapsed if elapsed > 0 else 0
        cv2.putText(frame, f"FPS: {fps:.1f}", (10, 60),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        # Show threshold on screen
        cv2.putText(frame, f"Threshold: {CONFIDENCE_THRESHOLD*100:.0f}%", (10, 85),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        
        # Display the frame
        cv2.imshow('ExamGuard - PHONE ONLY (Press q to quit)', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    camera.release()
    cv2.destroyAllWindows()
    print(f"Done! {frame_count} frames processed at {fps:.1f} FPS.")

## Cell 4: Experiment!

Go back to **Cell 2**, change the threshold, and re-run Cell 3:

| Try this | What you should see |
|----------|--------------------|
| `CONFIDENCE_THRESHOLD = 0.3` | More detections — maybe some false alarms (random objects labeled as phone) |
| `CONFIDENCE_THRESHOLD = 0.5` | Balanced — most real phones detected, few false alarms |
| `CONFIDENCE_THRESHOLD = 0.8` | Very strict — only very clear phones detected, might miss some |

### Questions to think about:

1. At what threshold does your phone start getting missed?
2. At what threshold do you start seeing false alarms?
3. For ExamGuard, what's worse: missing a real phone, or a false alarm?
   - Missing real phone = cheating goes undetected → BAD
   - False alarm = teacher walks over for nothing → annoying but not harmful
   - So we'd rather have LOWER threshold (catch more, accept some false alarms)

## Next Step

Task 05: Add alerts — beep sound + save screenshot when phone is detected.

Open: `../05_alert_system/README.md`